# Step 1: Setup and Import the JSON File
In this step we install the required libraries, upload our timetable data file (campus_data.json), and load it into Python so the assistant can use it.

Install and import libraries
This cell installs the Groq library (for the language model) and Gradio (for the chat interface). It also imports the standard Python modules we will use: json to read the data file, and datetime to work with dates and times.

In [ ]:
!pip install groq gradio google-api-python-client google-auth-oauthlib google-auth-httplib2 -q

import json
from datetime import datetime, timedelta

Upload the JSON file
This cell opens a file chooser. Select campus_data.json from your computer. The file is saved in the Colab session, and the cell prints the uploaded file name so we can confirm it worked. Note: uploaded files are deleted when the Colab runtime resets, so you may need to upload it again later.

In [ ]:
from google.colab import files

uploaded = files.upload()
json_name = list(uploaded.keys())[0]
print("Uploaded file:", json_name)

Saving campus_data.json to campus_data.json
Uploaded file: campus_data.json


Load the JSON into Python
This cell reads the uploaded file and converts it into a Python dictionary called DATA. The structure is: programs, then semesters, then courses. The cell then prints every degree program and the semesters available for it, to confirm that the data loaded correctly.

In [ ]:
with open(json_name, "r", encoding="utf-8") as f:
    DATA = json.load(f)

programs = DATA["programs"]

for code, prog in programs.items():
    print(code, "-", prog["program_name"], "| semesters:", list(prog["semesters"].keys()))

BSCS - BS Computer Science | semesters: ['1', '2', '3', '4', '5', '6', '7', '8']
BSIT - BS Information Technology | semesters: ['1', '2', '3', '4', '5', '6', '7', '8']
BSDS - BS Data Science | semesters: ['1']
BSSE - BS Software Engineering | semesters: ['1']
BSCYS - BS Cyber Security | semesters: ['1']
BSAI - BS Artificial Intelligence | semesters: ['1']


 Preview one semester
This cell shows how to read the data: it takes the BSIT program, semester 8, and prints each subject together with its instructor name(s). This is exactly the kind of lookup the chatbot will do when a student selects a degree and semester.

In [ ]:
courses = programs["BSIT"]["semesters"]["8"]["courses"]

for c in courses:
    print("-", c["course_title"], "|", ", ".join(c["instructors"]))

- Cyber Security | Muhammad Aslam
- Database Administration & Management | Fizzah Ishtiaq, Sonia Jamil
- Database Administration & Management - Lab | Qasim Niaz
- IT Elective 5 (Advanced Programming) | Allah Rakha
- Applied Computing Competency-2 (ACC-2) | Program Head will allocate teacher
- Final Year Project - 2 (FYP-2) | Supervisor


# Step 2: Data Helper Functions
# In this step we create small reusable functions that read our JSON data: list degrees and semesters, find subjects and their instructors, and convert timetable times into a format we can compare.

Settings and time helpers
This cell stores adjustable settings (class length, lab length, meeting length, office hours) and the current student profile (degree and semester). The timetable stores times like 01:30 without AM/PM, so timetable_to_minutes() treats any hour before 8 as PM. fmt_minutes() converts minutes back to a readable time such as 1:30 PM.

In [ ]:
import re
from zoneinfo import ZoneInfo

TZ = ZoneInfo("Asia/Karachi")

THEORY_MINUTES = 90          # assumed length of a lecture row
LAB_MINUTES = 180            # assumed length of a lab row
MEETING_MINUTES = 30         # length of a student-instructor meeting
OFFICE_START, OFFICE_END = "09:00", "16:00"   # 24-hour clock, adjust as you like

STUDENT = {"program": None, "semester": None}   # filled when the student chooses

def now():
    return datetime.now(TZ)

def clock_to_minutes(t):
    h, m = t.split(":")[:2]
    return int(h) * 60 + int(m)

def timetable_to_minutes(t):
    mins = clock_to_minutes(t)
    return mins + 12 * 60 if mins < 8 * 60 else mins    # 01:30 -> 13:30

def fmt_minutes(mins):
    h, m = divmod(mins, 60)
    return f"{h % 12 or 12}:{m:02d} {'AM' if h < 12 else 'PM'}"

def show(x):
    print(json.dumps(x, indent=2, ensure_ascii=False))

Degree, semester and subject lookups
These functions answer the first questions the chatbot asks. list_programs() and list_semesters() give the choices for the student. get_subjects() returns every subject of one semester with its instructor names (duplicates from different sections are merged). unique_instructors() returns the real instructor names and skips placeholders such as "Supervisor".

In [ ]:
PLACEHOLDER_TEACHERS = {"Supervisor", "Program Head will allocate teacher"}

def normalize_program(p):
    if not p:
        return None
    key = re.sub(r"[\s\-]", "", p.upper())
    for code, prog in programs.items():
        name = re.sub(r"[\s\-]", "", prog["program_name"].upper())
        if key == code or key == name or (len(key) >= 4 and key in name):
            return code
    return None

def list_programs():
    return [{"code": c, "name": p["program_name"]} for c, p in programs.items()]

def list_semesters(program):
    code = normalize_program(program)
    return [int(s) for s in programs[code]["semesters"]] if code else []

def get_semester_courses(program, semester):
    code = normalize_program(program)
    if not code:
        return []
    return programs[code]["semesters"].get(str(semester), {}).get("courses", [])

def get_subjects(program, semester):
    seen = {}
    for c in get_semester_courses(program, semester):
        item = seen.setdefault(c["course_title"], {"subject": c["course_title"], "instructors": []})
        for n in c["instructors"]:
            if n not in item["instructors"]:
                item["instructors"].append(n)
    return list(seen.values())

def format_subjects(program, semester):
    lines = []
    for i, s in enumerate(get_subjects(program, semester), 1):
        lines.append(f"{i}. {s['subject']} - {', '.join(s['instructors'])}")
    return "\n".join(lines)

def unique_instructors(program, semester):
    names = []
    for c in get_semester_courses(program, semester):
        for n in c["instructors"]:
            if n and n not in PLACEHOLDER_TEACHERS and n not in names:
                names.append(n)
    return names

Test the helpers
This cell prints the available degrees, the semesters of BSIT, and the subjects with instructors for BSIT semester 8. If the output looks right, the helpers work.

In [ ]:
print(list_programs())
print("BSIT semesters:", list_semesters("BSIT"))
print()
print(format_subjects("BSIT", 8))
print()
print("Instructors:", unique_instructors("BSIT", 8))

[{'code': 'BSCS', 'name': 'BS Computer Science'}, {'code': 'BSIT', 'name': 'BS Information Technology'}, {'code': 'BSDS', 'name': 'BS Data Science'}, {'code': 'BSSE', 'name': 'BS Software Engineering'}, {'code': 'BSCYS', 'name': 'BS Cyber Security'}, {'code': 'BSAI', 'name': 'BS Artificial Intelligence'}]
BSIT semesters: [1, 2, 3, 4, 5, 6, 7, 8]

1. Cyber Security - Muhammad Aslam
2. Database Administration & Management - Fizzah Ishtiaq, Sonia Jamil
3. Database Administration & Management - Lab - Qasim Niaz
4. IT Elective 5 (Advanced Programming) - Allah Rakha
5. Applied Computing Competency-2 (ACC-2) - Program Head will allocate teacher
6. Final Year Project - 2 (FYP-2) - Supervisor

Instructors: ['Muhammad Aslam', 'Fizzah Ishtiaq', 'Sonia Jamil', 'Qasim Niaz', 'Allah Rakha']


# Step 3: Campus Tools
These are the real Python functions that the AI will be allowed to call: get the class schedule, check lab availability, find course information, book and cancel meetings, and submit complaints. Meetings and complaints are stored in memory (mock storage), since this assignment does not need a real database.

Build a flat list of all class sessions
The JSON is organised by degree, semester and course. For schedule and lab questions it is easier to search one flat list. This cell loops through the whole JSON and creates SESSIONS, where each item has the day, start and end time (in minutes), room, teacher and course. It also builds LABS, the list of all lab rooms.

In [ ]:
SESSIONS = []
for code, prog in programs.items():
    for sem, semd in prog["semesters"].items():
        for c in semd["courses"]:
            is_lab = c["course_title"].strip().lower().endswith("lab")
            length = LAB_MINUTES if is_lab else THEORY_MINUTES
            for s in c["sessions"]:
                if not s["day"] or not s["time"]:
                    continue
                start = timetable_to_minutes(s["time"])
                SESSIONS.append({
                    "program": code, "semester": sem, "course": c["course_title"],
                    "section": c["section"], "teacher": s["teacher"], "day": s["day"],
                    "room": s["room"], "start": start, "end": start + length,
                })

LABS = sorted({s["room"] for s in SESSIONS if s["room"] and "LAB" in s["room"].upper()})
DAY_ORDER = {d: i for i, d in enumerate(
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])}

print(len(SESSIONS), "sessions loaded")
print(len(LABS), "labs:", LABS)

202 sessions loaded
13 labs: ['AFF-CYS-LAB-1', 'ASF-LAB-1', 'ASF-LAB-2', 'ASF-LAB-3', 'BFF-LAB-1', 'BGF1-LAB-1', 'BGF1-LAB-2', 'BGF1-LAB-3', 'BGF2-LAB-1', 'BGF2-LAB-2', 'BSF-LAB-1', 'BSF-LAB-2', 'BSF-LAB-3']


Tools 1 and 2 - schedule and course information
get_schedule() returns the student's classes for a given date (or weekday), sorted by time. It uses the degree and semester stored in STUDENT. get_course_info() searches the student's semester for a subject name and returns its instructor, days, times and rooms.

In [ ]:
def get_schedule(date=None, day=None):
    prog, sem = STUDENT["program"], STUDENT["semester"]
    if not prog or not sem:
        return {"error": "Please select your degree and semester first."}
    if date:
        try:
            day = datetime.strptime(date, "%Y-%m-%d").strftime("%A")
        except ValueError:
            return {"error": "Date must be in YYYY-MM-DD format."}
    rows = [s for s in SESSIONS
            if s["program"] == prog and s["semester"] == str(sem)
            and (day is None or s["day"].lower() == day.lower())]
    rows.sort(key=lambda s: (DAY_ORDER.get(s["day"].title(), 9), s["start"]))
    classes = [{"day": s["day"], "time": f"{fmt_minutes(s['start'])} - {fmt_minutes(s['end'])}",
                "course": s["course"], "instructor": s["teacher"], "room": s["room"]} for s in rows]
    return {"program": prog, "semester": sem, "day": day or "whole week",
            "classes": classes if classes else "No classes found for this day."}

def get_course_info(course_name):
    prog, sem = STUDENT["program"], STUDENT["semester"]
    if not prog or not sem:
        return {"error": "Please select your degree and semester first."}
    q = course_name.lower()
    rows = [s for s in SESSIONS if s["program"] == prog and s["semester"] == str(sem)
            and q in s["course"].lower()]
    if not rows:
        return {"error": f"No subject matching '{course_name}' in your semester.",
                "your_subjects": [x["subject"] for x in get_subjects(prog, sem)]}
    return {"matches": [{"course": s["course"], "instructor": s["teacher"], "day": s["day"],
                         "time": f"{fmt_minutes(s['start'])} - {fmt_minutes(s['end'])}",
                         "room": s["room"]} for s in rows]}

Tool 3 - check lab availability
Students say "Lab 2", but the real room names look like BGF1-LAB-2 or ASF-LAB-2. find_labs() matches the student's words to the real lab names. check_lab_availability() then looks at all timetable sessions in that lab on the chosen day. If a time is given, it says "available" or "busy" for that time. It always lists the busy time slots of the day.

In [ ]:
def find_labs(query):
    ql = re.sub(r"[\s\-]", "", query.upper())
    return [l for l in LABS if ql in re.sub(r"[\s\-]", "", l.upper())]

def check_lab_availability(lab, date, time=None):
    try:
        day = datetime.strptime(date, "%Y-%m-%d").strftime("%A")
    except ValueError:
        return {"error": "Date must be in YYYY-MM-DD format."}
    labs = find_labs(lab)
    if not labs:
        return {"error": f"No lab matches '{lab}'.", "known_labs": LABS}
    at = None
    if time:
        try:
            at = clock_to_minutes(time)
        except ValueError:
            return {"error": "Time must be in 24-hour HH:MM format."}
    results = []
    for l in labs:
        busy = sorted([s for s in SESSIONS if s["room"] == l and s["day"].lower() == day.lower()],
                      key=lambda s: s["start"])
        slots = [{"time": f"{fmt_minutes(s['start'])} - {fmt_minutes(s['end'])}",
                  "course": s["course"], "program": s["program"], "semester": s["semester"]} for s in busy]
        if at is None:
            status = "free except the busy slots listed"
        else:
            status = "busy" if any(s["start"] <= at < s["end"] for s in busy) else "available"
        results.append({"lab": l, "status": status, "busy_slots": slots})
    return {"date": date, "day": day, "time": fmt_minutes(at) if at is not None else None, "labs": results}

##Google Calendar Integration
When a meeting is booked, the assistant will create an event in the student's Google Calendar, and remove it when the meeting is cancelled. It will also read the calendar to avoid suggesting times that clash with existing events.

Upload the Google credentials
Upload credentials.json (OAuth client of type "Desktop app", Google Calendar API enabled). If you already have a token.json from your booking project, upload it too and the login step will be skipped.

In [ ]:
import os
from google.colab import files

print("Upload credentials.json (and token.json if you have it).")
up = files.upload()
for name, data in up.items():
    target = "token.json" if "token" in name.lower() else "credentials.json"
    with open(target, "wb") as f:
        f.write(data)
    print("Saved", target)

Upload credentials.json (and token.json if you have it).


Saving client_secret_520610253839-562sfml314ukpg6akf222t02tugop5pu.apps.googleusercontent.com.json to client_secret_520610253839-562sfml314ukpg6akf222t02tugop5pu.apps.googleusercontent.com.json
Saved credentials.json


Log in to Google Calendar
This cell loads token.json if it exists. Otherwise it prints a login link. Open it, allow access, and the browser will end on a page that fails to load (localhost refused to connect). That is normal: copy the full address from the address bar and paste it back here. The login is saved to token.json for next time.

In [ ]:
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from google_auth_oauthlib.flow import Flow
from googleapiclient.discovery import build

os.environ["OAUTHLIB_INSECURE_TRANSPORT"] = "1"
SCOPES = ["https://www.googleapis.com/auth/calendar"]
calendar_service = None
creds = None

if os.path.exists("token.json"):
    try:
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
    except Exception as e:
        print("Old token.json could not be used, logging in again:", e)
        creds = None

if not creds or not creds.valid:
    flow = Flow.from_client_secrets_file("credentials.json", scopes=SCOPES,
                                         redirect_uri="http://localhost:1")
    auth_url, _ = flow.authorization_url(prompt="consent", access_type="offline")
    print("1) Open this link, sign in and allow access:\n")
    print(auth_url)
    print("\n2) The next page will fail to load. Copy the FULL address from the address bar.")
    redirect_response = input("3) Paste the full URL here: ").strip()
    flow.fetch_token(authorization_response=redirect_response)
    creds = flow.credentials
    with open("token.json", "w") as f:
        f.write(creds.to_json())

calendar_service = build("calendar", "v3", credentials=creds)
print("Connected to calendar:", calendar_service.calendarList().get(calendarId="primary").execute()["summary"])

1) Open this link, sign in and allow access:

https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=520610253839-562sfml314ukpg6akf222t02tugop5pu.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A1&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcalendar&state=l60FZpxk2BVSO4Eyd5Ac32YOFXB8ul&code_challenge=bdhpBQIPOMswpA3h1HEvQCvR3L2Thik2kaXPTUf9bfw&code_challenge_method=S256&prompt=consent&access_type=offline

2) The next page will fail to load. Copy the FULL address from the address bar.
3) Paste the full URL here: http://localhost:1/?state=l60FZpxk2BVSO4Eyd5Ac32YOFXB8ul&iss=https://accounts.google.com&code=4/0AUn6Hd_BNHWn75Q_X7M8VHE2guwDGwjHHWreJY6yRUa9HBMTZJp6Oz-WVZ4KyYvhlfyyCA&scope=https://www.googleapis.com/auth/calendar
Connected to calendar: silent5worker363@gmail.com


##Calendar helper functions
calendar_create_event() adds a meeting to the calendar, calendar_delete_event() removes it, and calendar_busy() returns the busy time ranges of one day (in minutes) so free slots can avoid clashes. If the calendar is not connected, these functions return None or an empty list, so the rest of the assistant still works. Change STUDENT_NAME to your name; it appears in the event title.

In [ ]:
STUDENT_NAME = "Student"          # change to your name

def calendar_create_event(summary, date, start_min, end_min, description=""):
    if calendar_service is None:
        return None
    def iso(m):
        return f"{date}T{m // 60:02d}:{m % 60:02d}:00"
    body = {"summary": summary, "description": description,
            "start": {"dateTime": iso(start_min), "timeZone": "Asia/Karachi"},
            "end": {"dateTime": iso(end_min), "timeZone": "Asia/Karachi"}}
    try:
        return calendar_service.events().insert(calendarId="primary", body=body).execute()
    except Exception as e:
        return {"error": str(e)}

def calendar_delete_event(event_id):
    if calendar_service is None or not event_id:
        return False
    try:
        calendar_service.events().delete(calendarId="primary", eventId=event_id).execute()
        return True
    except Exception:
        return False

def calendar_busy(date):
    """Busy ranges of one day as a list of (start_minute, end_minute)."""
    if calendar_service is None:
        return []
    try:
        body = {"timeMin": f"{date}T00:00:00+05:00", "timeMax": f"{date}T23:59:59+05:00",
                "timeZone": "Asia/Karachi", "items": [{"id": "primary"}]}
        res = calendar_service.freebusy().query(body=body).execute()
        out = []
        for b in res["calendars"]["primary"]["busy"]:
            s = datetime.fromisoformat(b["start"].replace("Z", "+00:00")).astimezone(TZ)
            e = datetime.fromisoformat(b["end"].replace("Z", "+00:00")).astimezone(TZ)
            out.append((s.hour * 60 + s.minute, e.hour * 60 + e.minute if e.date() == s.date() else 24 * 60))
        return out
    except Exception:
        return []

 ##Tools 4, 5 and 6 - meetings (with Google Calendar)
slot_conflict() checks whether a 30-minute slot is free: the instructor must not be teaching, the student must not already have a meeting, and the Google Calendar must be empty. book_meeting() validates the request, creates the Google Calendar event and stores the meeting. cancel_meeting() cancels by ID (or the only scheduled meeting) and deletes the calendar event.

In [ ]:
MEETINGS = []

def public_meeting(m):
    return {k: m.get(k) for k in ("meeting_id", "instructor", "date", "time", "purpose", "status")}

def resolve_instructor(query):
    prog, sem = STUDENT["program"], STUDENT["semester"]
    if prog and sem:
        names = unique_instructors(prog, sem)
    else:
        names = sorted({s["teacher"] for s in SESSIONS if s["teacher"] and s["teacher"] not in PLACEHOLDER_TEACHERS})
    q = query.lower()
    found = [n for n in names if q in n.lower() or n.lower() in q]
    if not found and prog and sem:
        for c in get_semester_courses(prog, sem):
            if q in c["course_title"].lower():
                found += [n for n in c["instructors"] if n not in PLACEHOLDER_TEACHERS and n not in found]
    return found, names

def slot_conflict(inst, date, day, start, end, busy=None):
    """Return a reason (string) if the slot is NOT free, otherwise None."""
    for s in SESSIONS:
        if s["teacher"] == inst and s["day"].lower() == day.lower() and s["start"] < end and start < s["end"]:
            return (f"{inst} is teaching {s['course']} at that time "
                    f"({fmt_minutes(s['start'])} - {fmt_minutes(s['end'])}).")
    for m in MEETINGS:
        if m["status"] == "Scheduled" and m["date"] == date and m["start"] < end and start < m["end"]:
            if m["instructor"] == inst:
                return f"{inst} already has a meeting at that time."
            return f"You already have a meeting with {m['instructor']} at that time."
    for b0, b1 in (busy or []):
        if b0 < end and start < b1:
            return "Your Google Calendar already has an event at that time."
    return None

def book_meeting(instructor, date, time, purpose="General discussion"):
    found, names = resolve_instructor(instructor)
    if not found:
        return {"error": f"No instructor matches '{instructor}'.", "available_instructors": names}
    if len(found) > 1:
        return {"needs_clarification": "Which instructor do you mean?", "matches": found}
    inst = found[0]
    try:
        when = datetime.strptime(f"{date} {time}", "%Y-%m-%d %H:%M").replace(tzinfo=TZ)
    except ValueError:
        return {"error": "Use date YYYY-MM-DD and time HH:MM (24-hour)."}
    day = when.strftime("%A")
    start = clock_to_minutes(time)
    end = start + MEETING_MINUTES
    if day == "Sunday":
        return {"error": "Meetings are not available on Sunday."}
    if when <= now():
        return {"error": "That time is in the past. Please choose a future time."}
    if start < clock_to_minutes(OFFICE_START) or end > clock_to_minutes(OFFICE_END):
        return {"error": f"Meetings are only possible between {OFFICE_START} and {OFFICE_END}."}
    reason = slot_conflict(inst, date, day, start, end, calendar_busy(date))
    if reason:
        return {"error": reason + " Please pick another time."}

    event = calendar_create_event(f"Meeting with {inst} ({STUDENT_NAME})", date, start, end,
                                  f"Purpose: {purpose}")
    ok = bool(event and "id" in event)
    meeting = {"meeting_id": f"M{len(MEETINGS) + 1:03d}", "instructor": inst, "date": date,
               "time": f"{fmt_minutes(start)} - {fmt_minutes(end)}", "purpose": purpose,
               "status": "Scheduled", "start": start, "end": end,
               "event_id": event["id"] if ok else None}
    MEETINGS.append(meeting)
    cal = ({"added": True, "link": event.get("htmlLink")} if ok else
           {"added": False, "reason": (event or {}).get("error", "Google Calendar is not connected.")})
    return {"booked": True, "meeting": public_meeting(meeting), "google_calendar": cal}

def list_meetings():
    return {"meetings": [public_meeting(m) for m in MEETINGS] or "You have no meetings."}

def cancel_meeting(meeting_id=None):
    active = [m for m in MEETINGS if m["status"] == "Scheduled"]
    if meeting_id:
        target = next((m for m in MEETINGS if m["meeting_id"].lower() == meeting_id.lower()), None)
        if not target:
            return {"error": f"No meeting with ID {meeting_id}."}
        if target["status"] == "Cancelled":
            return {"error": "That meeting is already cancelled."}
    else:
        if not active:
            return {"error": "You have no scheduled meetings to cancel."}
        if len(active) > 1:
            return {"needs_clarification": "You have several scheduled meetings. Which one?",
                    "scheduled_meetings": [public_meeting(m) for m in active]}
        target = active[0]
    removed = calendar_delete_event(target.get("event_id"))
    target["status"] = "Cancelled"
    return {"cancelled": True, "meeting": public_meeting(target), "calendar_event_removed": removed}

##Tool 7 - complaints (saved to complaints.json)
submit_complaint() saves a complaint with a unique ID, category, description and time. Every complaint is written to complaints.json, so it is kept even after the app restarts (as long as the Colab session stays alive). Existing complaints are loaded when the cell runs.

In [ ]:
COMPLAINTS_FILE = "complaints.json"
COMPLAINT_CATEGORIES = ["Internet/WiFi", "Library", "Hostel", "Transport",
                        "Classroom/Lab Equipment", "Cafeteria", "Other"]

def load_complaints():
    if os.path.exists(COMPLAINTS_FILE):
        with open(COMPLAINTS_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return []

def save_complaints():
    with open(COMPLAINTS_FILE, "w", encoding="utf-8") as f:
        json.dump(COMPLAINTS, f, indent=2, ensure_ascii=False)

COMPLAINTS = load_complaints()

def submit_complaint(category, description):
    if not description or len(description.strip()) < 10:
        return {"error": "Please describe the problem in a little more detail (at least 10 characters)."}
    q = (category or "").lower()
    cat = next((c for c in COMPLAINT_CATEGORIES
                if q and (q in c.lower() or c.lower().split("/")[0] in q)), "Other")
    complaint = {"complaint_id": f"C{len(COMPLAINTS) + 1:03d}", "category": cat,
                 "description": description.strip(), "status": "Submitted",
                 "submitted_at": now().strftime("%Y-%m-%d %H:%M"),
                 "program": STUDENT["program"], "semester": STUDENT["semester"]}
    COMPLAINTS.append(complaint)
    save_complaints()
    return {"submitted": True, "complaint_id": complaint["complaint_id"], "category": cat,
            "status": "Submitted"}

Tool 8 - instructor free slots
get_instructor_free_slots() lists every 30-minute slot on a date when the instructor can meet: inside office hours, not during the instructor's classes, not already booked, and free in the student's Google Calendar. The Meeting tab and the chatbot both use it.

In [ ]:
def get_instructor_free_slots(instructor, date):
    found, names = resolve_instructor(instructor)
    if not found:
        return {"error": f"No instructor matches '{instructor}'.", "available_instructors": names}
    if len(found) > 1:
        return {"needs_clarification": "Which instructor do you mean?", "matches": found}
    inst = found[0]
    try:
        d = datetime.strptime(date, "%Y-%m-%d")
    except ValueError:
        return {"error": "Date must be in YYYY-MM-DD format."}
    day = d.strftime("%A")
    if day == "Sunday":
        return {"error": "Meetings are not available on Sunday."}
    if d.date() < now().date():
        return {"error": "That date is in the past."}

    busy = calendar_busy(date)
    now_min = now().hour * 60 + now().minute if d.date() == now().date() else -1
    slots, t = [], clock_to_minutes(OFFICE_START)
    while t + MEETING_MINUTES <= clock_to_minutes(OFFICE_END):
        if t > now_min and not slot_conflict(inst, date, day, t, t + MEETING_MINUTES, busy):
            slots.append({"start": f"{t // 60:02d}:{t % 60:02d}",
                          "label": f"{fmt_minutes(t)} - {fmt_minutes(t + MEETING_MINUTES)}"})
        t += MEETING_MINUTES
    teaching = [f"{s['course']} {fmt_minutes(s['start'])} - {fmt_minutes(s['end'])}"
                for s in sorted(SESSIONS, key=lambda s: s["start"])
                if s["teacher"] == inst and s["day"].lower() == day.lower()]
    return {"instructor": inst, "date": date, "day": day, "free_slots": slots,
            "teaching_that_day": teaching,
            "message": "" if slots else "No free slots on this date. Try another day."}

##Test all tools without the AI
We set the student as BSIT semester 8 and test every tool directly. The meeting test picks the first real free slot of the first instructor, books it (a real event appears in your Google Calendar), lists it, and then cancels it (the event is removed).

In [ ]:
STUDENT["program"], STUDENT["semester"] = "BSIT", "8"

def next_weekday(target):          # Monday = 0 ... Sunday = 6
    d = now() + timedelta(days=1)
    while d.weekday() != target:
        d += timedelta(days=1)
    return d.strftime("%Y-%m-%d")

tomorrow = (now() + timedelta(days=1)).strftime("%Y-%m-%d")
wednesday = next_weekday(2)
inst = unique_instructors("BSIT", 8)[0]

print("--- Schedule tomorrow"); show(get_schedule(tomorrow))
print("--- Lab 2 at 12:00 tomorrow"); show(check_lab_availability("Lab 2", tomorrow, "12:00"))
print("--- Course info"); show(get_course_info("Cyber"))
print("--- Free slots", inst); slots = get_instructor_free_slots(inst, wednesday); show(slots)
first = slots["free_slots"][0]["start"]
print("--- Book meeting"); show(book_meeting(inst, wednesday, first, "FYP discussion"))
print("--- List meetings"); show(list_meetings())
print("--- Cancel previous meeting"); show(cancel_meeting())
print("--- Complaint"); show(submit_complaint("Internet", "The WiFi in the library building is very slow."))

--- Schedule tomorrow
{
  "program": "BSIT",
  "semester": "8",
  "day": "Wednesday",
  "classes": [
    {
      "day": "Wednesday",
      "time": "9:00 AM - 10:30 AM",
      "course": "IT Elective 5 (Advanced Programming)",
      "instructor": "Allah Rakha",
      "room": "BSF-6"
    },
    {
      "day": "Wednesday",
      "time": "10:30 AM - 12:00 PM",
      "course": "IT Elective 5 (Advanced Programming)",
      "instructor": "Allah Rakha",
      "room": "BSF-6"
    }
  ]
}
--- Lab 2 at 12:00 tomorrow
{
  "date": "2026-11-18",
  "day": "Wednesday",
  "time": "12:00 PM",
  "labs": [
    {
      "lab": "ASF-LAB-2",
      "status": "available",
      "busy_slots": []
    },
    {
      "lab": "BGF1-LAB-2",
      "status": "busy",
      "busy_slots": [
        {
          "time": "11:00 AM - 2:00 PM",
          "course": "Domain Elective -3 (Data Warehousing & Business Intelligence) - Lab",
          "program": "BSIT",
          "semester": "6"
        }
      ]
    },
    {
      "lab

In [ ]:
STUDENT["program"], STUDENT["semester"] = "BSIT", "8"

def next_weekday(target):          # Monday = 0 ... Sunday = 6
    d = now() + timedelta(days=1)
    while d.weekday() != target:
        d += timedelta(days=1)
    return d.strftime("%Y-%m-%d")

tomorrow = (now() + timedelta(days=1)).strftime("%Y-%m-%d")
wednesday = next_weekday(2)

print("--- Schedule tomorrow"); show(get_schedule(tomorrow))
print("--- Lab 2 at 12:00 tomorrow"); show(check_lab_availability("Lab 2", tomorrow, "12:00"))
print("--- Course info"); show(get_course_info("Cyber"))
print("--- Book meeting"); show(book_meeting("Qasim Niaz", wednesday, "10:00", "FYP discussion"))
print("--- List meetings"); show(list_meetings())
print("--- Cancel previous meeting"); show(cancel_meeting())
print("--- Complaint"); show(submit_complaint("Internet", "The WiFi in the library building is very slow."))

--- Schedule tomorrow
{
  "program": "BSIT",
  "semester": "8",
  "day": "Wednesday",
  "classes": [
    {
      "day": "Wednesday",
      "time": "9:00 AM - 10:30 AM",
      "course": "IT Elective 5 (Advanced Programming)",
      "instructor": "Allah Rakha",
      "room": "BSF-6"
    },
    {
      "day": "Wednesday",
      "time": "10:30 AM - 12:00 PM",
      "course": "IT Elective 5 (Advanced Programming)",
      "instructor": "Allah Rakha",
      "room": "BSF-6"
    }
  ]
}
--- Lab 2 at 12:00 tomorrow
{
  "date": "2026-11-18",
  "day": "Wednesday",
  "time": "12:00 PM",
  "labs": [
    {
      "lab": "ASF-LAB-2",
      "status": "available",
      "busy_slots": []
    },
    {
      "lab": "BGF1-LAB-2",
      "status": "busy",
      "busy_slots": [
        {
          "time": "11:00 AM - 2:00 PM",
          "course": "Domain Elective -3 (Data Warehousing & Business Intelligence) - Lab",
          "program": "BSIT",
          "semester": "6"
        }
      ]
    },
    {
      "lab

# Step 4: Groq LLM Setup and Tool Definitions
In this step we connect to the Groq language model and describe our seven campus tools to it in a format it understands (function calling). The model never runs the tools itself. It only chooses a tool name and the arguments, and our Python code runs the real function.

Connect to Groq
This cell reads the Groq API key from Colab Secrets (the key icon on the left; the secret must be named GROQ_API_KEY and "Notebook access" must be switched on for this notebook). It creates the Groq client and selects the model. It then sends one short test message to confirm the connection works. If the model name is rejected, run the second block to list the models available in your account and pick another chat model.

In [ ]:
from google.colab import userdata
from groq import Groq

client = Groq(api_key=userdata.get("GROQ_API_KEY"))
MODEL = "openai/gpt-oss-120b"      # fallback: "openai/gpt-oss-20b"

r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say hello in one short sentence."}],
)
print(r.choices[0].message.content)

# If the model name is not found, uncomment the next line to see the available models:
# print(sorted(m.id for m in client.models.list().data))

Hello!


 Describe the tools to the model
Each tool is described by its name, a plain-English description of when to use it, and its parameters. The model reads these descriptions to decide which tool fits the student's request. TOOL_FUNCTIONS connects every tool name to the real Python function from Step 3. run_tool() is a safe wrapper: it rejects unknown tool names, removes arguments the function does not accept, and turns any error into a message instead of crashing.

In [ ]:
import inspect

def S(desc):
    return {"type": "string", "description": desc}

def tool(name, description, properties, required=()):
    return {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object", "properties": properties, "required": list(required)}}}

TOOLS = [
    tool("get_schedule",
         "Get the student's class schedule for one date or one weekday. Use for questions like 'what is my schedule tomorrow'.",
         {"date": S("Date in YYYY-MM-DD format"),
          "day": S("Weekday name such as Monday, only if no date is given")}),
    tool("get_course_info",
         "Find the instructor, days, times and rooms of one subject in the student's semester.",
         {"course_name": S("Subject name or part of it, e.g. 'Cyber Security'")}, ["course_name"]),
    tool("check_lab_availability",
         "Check whether a lab is available on a date, optionally at a specific time.",
         {"lab": S("Lab name as the student says it, e.g. 'Lab 2'"),
          "date": S("Date in YYYY-MM-DD format"),
          "time": S("24-hour time HH:MM, only if the student gave a time")}, ["lab", "date"]),
    tool("book_meeting",
         "Book a 30-minute meeting with an instructor. Only call it when instructor, date and time are all known.",
         {"instructor": S("Instructor name, or the subject they teach"),
          "date": S("Date in YYYY-MM-DD format"),
          "time": S("24-hour time HH:MM"),
          "purpose": S("Reason for the meeting")}, ["instructor", "date", "time"]),
    tool("list_meetings",
         "List all of the student's meetings with their IDs and status.", {}),
    tool("cancel_meeting",
         "Cancel a meeting. If the student says 'my previous' or 'my last' meeting, call this WITHOUT meeting_id.",
         {"meeting_id": S("Meeting ID such as M001, only if the student gave one")}),
    tool("submit_complaint",
         "Submit a campus complaint such as internet, library, hostel, transport, equipment or cafeteria problems.",
         {"category": S("Short category, e.g. 'Internet'"),
          "description": S("What the problem is, in the student's words")}, ["category", "description"]),
]

TOOL_FUNCTIONS = {
    "get_schedule": get_schedule,
    "get_course_info": get_course_info,
    "check_lab_availability": check_lab_availability,
    "book_meeting": book_meeting,
    "list_meetings": list_meetings,
    "cancel_meeting": cancel_meeting,
    "submit_complaint": submit_complaint,
}

def run_tool(name, args):
    fn = TOOL_FUNCTIONS.get(name)
    if fn is None:
        return {"error": f"Unknown tool '{name}'."}
    allowed = inspect.signature(fn).parameters
    clean = {k: v for k, v in args.items() if k in allowed}
    try:
        return fn(**clean)
    except TypeError as e:
        return {"error": f"Missing or wrong arguments: {e}"}
    except Exception as e:
        return {"error": f"Tool failed: {e}"}

print(len(TOOLS), "tools registered:", [t["function"]["name"] for t in TOOLS])

7 tools registered: ['get_schedule', 'get_course_info', 'check_lab_availability', 'book_meeting', 'list_meetings', 'cancel_meeting', 'submit_complaint']


##Register the instructor availability tool
This adds an eighth tool, get_instructor_availability, so the chatbot can answer "When is Sir free on Wednesday?" using the real free slots.

In [ ]:
TOOLS[:] = [t for t in TOOLS if t["function"]["name"] != "get_instructor_availability"]
TOOLS.append(tool(
    "get_instructor_availability",
    "List the free 30-minute meeting slots of an instructor on a date. Use when the student asks when an instructor is free, or before booking a meeting.",
    {"instructor": S("Instructor name, or the subject they teach"),
     "date": S("Date in YYYY-MM-DD format")}, ["instructor", "date"]))
TOOL_FUNCTIONS["get_instructor_availability"] = get_instructor_free_slots

print(len(TOOLS), "tools registered:", [t["function"]["name"] for t in TOOLS])

8 tools registered: ['get_schedule', 'get_course_info', 'check_lab_availability', 'book_meeting', 'list_meetings', 'cancel_meeting', 'submit_complaint', 'get_instructor_availability']


##STEP 5: The Function-Calling Loop
This step connects everything. The student writes a message, the model decides whether a tool is needed, our code runs the tool, the result goes back to the model, and the model writes the final answer. This repeats until the model has no more tools to call.

##The system prompt
The system prompt gives the assistant its rules. It is rebuilt on every message so it contains today's date, the next eight days, and the student's degree and semester. A new rule tells the model to use get_instructor_availability when the student asks when an instructor is free.

In [ ]:
def build_system_prompt():
    today = now()
    upcoming = ", ".join((today + timedelta(days=i)).strftime("%A %Y-%m-%d") for i in range(8))
    prog, sem = STUDENT["program"], STUDENT["semester"]
    student = (f"{programs[prog]['program_name']} ({prog}), semester {sem}"
               if prog and sem else "not selected yet")
    return f"""You are the Smart Campus Assistant for BS students of the university.
Today is {today.strftime('%A, %Y-%m-%d')} (Pakistan time). Next days: {upcoming}.
The student is in: {student}.

RULES
- Use the tools for every fact about schedules, subjects, instructors, labs, meetings and complaints. Never invent data.
- Convert words like "tomorrow" or "next Monday" into YYYY-MM-DD using the dates above. Times for tools are 24-hour HH:MM.
- If required information is missing (for example the instructor, date or time for a meeting), ask ONE short question instead of guessing.
- If the student asks when an instructor is free, call get_instructor_availability and list the free slots. Book a meeting only after the student has chosen instructor, date and time. Booked meetings are added to the student's Google Calendar.
- "Cancel my previous meeting" means: call cancel_meeting without a meeting_id.
- If a tool result contains "needs_clarification" or an error, explain it simply and ask the student what to do next.
- Only help with campus tasks. For anything else, politely say you can only help with campus tasks.
- Keep answers short, friendly and in English. Use short lists for schedules."""

 The assistant function
ask_assistant() runs one conversation turn. It sends the system prompt, the earlier messages (history) and the new message to the model with the tool list. If the model asks for tools, the function runs each one with run_tool(), adds the results to the conversation and asks the model again (up to six rounds). If Groq rejects a malformed tool call, it retries once. At the end it saves the question and the answer in history, so the assistant remembers the conversation. With show_calls=True it prints which tool was called, which is useful to demonstrate function calling.

In [ ]:
def ask_assistant(user_message, history, show_calls=True):
    messages = ([{"role": "system", "content": build_system_prompt()}]
                + history + [{"role": "user", "content": user_message}])
    answer, failures = None, 0

    for _ in range(6):
        try:
            r = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS)
        except Exception as e:
            failures += 1
            if failures >= 2:
                answer = "Sorry, I had a problem understanding that. Please rephrase your request."
                break
            continue

        msg = r.choices[0].message
        if not msg.tool_calls:
            answer = msg.content or "Sorry, I could not answer that."
            break

        messages.append({
            "role": "assistant", "content": msg.content or "",
            "tool_calls": [{"id": tc.id, "type": "function",
                            "function": {"name": tc.function.name,
                                         "arguments": tc.function.arguments}}
                           for tc in msg.tool_calls]})
        for tc in msg.tool_calls:
            try:
                args = json.loads(tc.function.arguments or "{}")
            except Exception:
                args = {}
            if show_calls:
                print(f"   [tool called] {tc.function.name}({args})")
            result = run_tool(tc.function.name, args)
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(result, ensure_ascii=False)})

    if answer is None:
        answer = "Sorry, I could not finish that request. Please try again."
    history.append({"role": "user", "content": user_message})
    history.append({"role": "assistant", "content": answer})
    return answer

Test the loop with two questions
This cell selects BSIT semester 8 as the student, creates an empty conversation history, and asks two of the assignment's example questions. The printed lines starting with [tool called] show that the model chose the correct tool and the correct arguments by itself.

In [ ]:
STUDENT["program"], STUDENT["semester"] = "BSIT", "8"
history = []

for q in ["What is the schedule for tomorrow?",
          "Check whether Lab 2 is available tomorrow at 12:00."]:
    print("STUDENT:", q)
    print("ASSISTANT:", ask_assistant(q, history))
    print("-" * 60)

STUDENT: What is the schedule for tomorrow?
   [tool called] get_schedule({'date': '2026-11-18'})
ASSISTANT: Here’s your schedule for **Wednesday, 2026‑11‑18**:

- **09:00 – 10:30** – IT Elective 5 (Advanced Programming) – Instructor: Allah Rakha – Room BSF‑6  
- **10:30 – 12:00** – IT Elective 5 (Advanced Programming) – Instructor: Allah Rakha – Room BSF‑6  

Let me know if you need anything else!
------------------------------------------------------------
STUDENT: Check whether Lab 2 is available tomorrow at 12:00.
   [tool called] check_lab_availability({'date': '2026-11-18', 'lab': 'Lab 2', 'time': '12:00'})
ASSISTANT: Lab 2 has several locations. At **12:00 on 2026‑11‑18**:

- **ASF‑LAB‑2** – available  
- **BGF1‑LAB‑2** – busy (Data Warehousing & BI lab)  
- **BGF2‑LAB‑2** – available  
- **BSF‑LAB‑2** – busy (Programming Fundamentals lab)

Which Lab 2 would you like to use?
------------------------------------------------------------


Recommended Questions
After the student chooses a degree and semester, the chatbot will show suggested questions. In this step we generate them from the real data, so they use the student's own subjects, instructors and labs. In Step 7 these will become clickable buttons.

Generate the recommended questions
recommended_questions() builds up to six questions for a given degree and semester: tomorrow's schedule, who teaches the first subject, a lab availability check (using a lab that this semester actually uses), a meeting with a real instructor, cancelling the previous meeting, and a complaint about the internet. These cover all five tasks of the assignment.

In [ ]:
def lab_label(program, semester):
    for s in SESSIONS:
        if s["program"] == program and s["semester"] == str(semester) \
           and s["room"] and "LAB" in s["room"].upper():
            m = re.search(r"LAB-(\d+)", s["room"].upper())
            if m:
                return f"Lab {m.group(1)}"
    return "Lab 1"

def recommended_questions(program, semester, n=6):
    subjects = [s for s in get_subjects(program, semester)
                if not s["subject"].strip().lower().endswith("lab")]
    instructors = unique_instructors(program, semester)
    qs = ["What is my schedule for tomorrow?"]
    if subjects:
        qs.append(f"Who teaches {subjects[0]['subject']} and when is the class?")
    qs.append(f"Check whether {lab_label(program, semester)} is available tomorrow at 11:00.")
    if instructors:
        qs.append(f"Book a meeting with {instructors[0]} on Wednesday at 10:00.")
    qs.append("Cancel my previous meeting.")
    qs.append("I want to submit a complaint about the internet.")
    return qs[:n]

Test the recommended questions
This cell prints the recommended questions for two different semesters to show that they change with the student's degree and semester. It then runs the first recommended question through the assistant to confirm the questions work end to end.

In [ ]:
for p, s in [("BSIT", 8), ("BSCS", 3)]:
    print(f"--- {p} semester {s}")
    for q in recommended_questions(p, s):
        print(" *", q)

print()
STUDENT["program"], STUDENT["semester"] = "BSIT", "8"
history = []
qs = recommended_questions("BSIT", 8)
print("STUDENT:", qs[1])
print("ASSISTANT:", ask_assistant(qs[1], history))

--- BSIT semester 8
 * What is my schedule for tomorrow?
 * Who teaches Cyber Security and when is the class?
 * Check whether Lab 1 is available tomorrow at 11:00.
 * Book a meeting with Muhammad Aslam on Wednesday at 10:00.
 * Cancel my previous meeting.
 * I want to submit a complaint about the internet.
--- BSCS semester 3
 * What is my schedule for tomorrow?
 * Who teaches Data Structure and when is the class?
 * Check whether Lab 2 is available tomorrow at 11:00.
 * Book a meeting with Muhammad Haris on Wednesday at 10:00.
 * Cancel my previous meeting.
 * I want to submit a complaint about the internet.

STUDENT: Who teaches Cyber Security and when is the class?
   [tool called] get_course_info({'course_name': 'Cyber Security'})
ASSISTANT: **Course:** Cyber Security  
**Instructor:** Muhammad Aslam  
**Day:** Thursday  
**Time:**  
- 12:00 PM – 1:30 PM (Room BSF‑6)  
- 1:30 PM – 3:00 PM (Room BSF‑6)  

Let me know if you need anything else, like the next class date or to book a 

## Step 7: Academic Design and Gradio Interface
In this final step we build the user interface. We first design the look of the application with HTML and CSS (an academic style with navy and gold colours), and then connect it to Gradio. The student selects a degree, then a semester, then sees the subjects with instructor names, then picks a recommended question or types their own, and chats with the assistant.

# The HTML/CSS design
This cell defines the visual style: navy and gold theme, serif headings, banner, progress bar, degree and semester buttons, subjects table, and the new styles for tabs, the weekly schedule day-cards, free (green) and busy (red) time slots, and message boxes. The table colours use !important so Gradio's dark theme cannot override them. The last line previews the banner.

In [ ]:
from IPython.display import HTML, display

CSS = """
<style>
.gradio-container {max-width: 1000px !important; margin: auto !important;}
footer {display: none !important;}

.hero {background: linear-gradient(135deg, #0B2545 0%, #13315C 60%, #1d4e89 100%);
       color: #fff; padding: 28px 30px; border-radius: 18px; margin-bottom: 12px;
       border-bottom: 4px solid #C9A227; box-shadow: 0 10px 28px rgba(11,37,69,.35);}
.hero h1 {margin: 0; font-family: Georgia, 'Times New Roman', serif; font-size: 30px;
          letter-spacing: .3px; color: #fff !important;}
.hero p {margin: 8px 0 0; color: #dbe5f1 !important; line-height: 1.5;}

.stepper {display: flex; justify-content: space-between; gap: 6px; margin: 6px 0 12px;}
.step {flex: 1; text-align: center; font-size: 12px; opacity: .55;}
.step .dot {display: inline-flex; width: 30px; height: 30px; border-radius: 50%;
            align-items: center; justify-content: center; font-weight: 700;
            background: rgba(128,128,128,.35); margin-bottom: 4px;}
.step .lbl {display: block;}
.step.active {opacity: 1; font-weight: 700;}
.step.active .dot {background: #0B2545; color: #fff; box-shadow: 0 0 0 4px rgba(201,162,39,.55);}
.step.done {opacity: 1;}
.step.done .dot {background: #C9A227; color: #0B2545;}

.section-title {background: #F8F5EE !important; color: #0B2545 !important; padding: 11px 16px; margin: 6px 0 10px;
                border-left: 5px solid #C9A227; border-radius: 8px;
                font-family: Georgia, 'Times New Roman', serif; font-size: 18px;}

.deg-btn {background: linear-gradient(135deg, #0B2545, #1d4e89) !important; color: #fff !important;
          border: none !important; border-left: 5px solid #C9A227 !important; border-radius: 12px !important;
          padding: 18px 14px !important; font-weight: 600 !important; text-align: left !important;
          transition: all .15s ease;}
.deg-btn:hover {transform: translateY(-2px); box-shadow: 0 8px 18px rgba(11,37,69,.4);}

.sem-btn {background: #F8F5EE !important; color: #0B2545 !important; border: 2px solid #C9A227 !important;
          border-radius: 999px !important; font-weight: 700 !important; transition: all .15s ease;}
.sem-btn:hover {background: #C9A227 !important; transform: translateY(-2px);}

.q-btn {background: #fff !important; color: #0B2545 !important; border: 1px solid #d9cfae !important;
        border-radius: 12px !important; text-align: left !important; font-weight: 500 !important;
        transition: all .15s ease;}
.q-btn:hover {border-color: #C9A227 !important; background: #fffaf0 !important; transform: translateX(3px);}

.go-btn {background: #0B2545 !important; color: #fff !important; border: none !important;
         border-bottom: 3px solid #C9A227 !important; border-radius: 10px !important; font-weight: 700 !important;}
.go-btn:hover {background: #1d4e89 !important;}
.warn-btn {background: #fff !important; color: #842029 !important; border: 1px solid #e59a9a !important;
           border-radius: 10px !important; font-weight: 600 !important;}

.link-btn {background: transparent !important; color: #C9A227 !important; border: none !important;
           text-decoration: underline; box-shadow: none !important;}

/* Tabs */
button[role="tab"] {font-weight: 600 !important; font-size: 15px !important;}
button[role="tab"][aria-selected="true"], button[role="tab"].selected {
    color: #C9A227 !important; border-bottom: 3px solid #C9A227 !important;}

/* Cards and tables */
.card {background: #fff; color: #1b2733; border-radius: 14px; margin: 6px 0 14px;
       box-shadow: 0 4px 16px rgba(0,0,0,.18); overflow: hidden;}
.card-head, .card-head span {color: #ffffff !important;}
.card-head {background: #0B2545; padding: 14px 18px; border-bottom: 3px solid #C9A227;
            font-family: Georgia, 'Times New Roman', serif; font-size: 18px;
            display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 8px;}
.badge {background: #C9A227 !important; color: #0B2545 !important; padding: 3px 12px; border-radius: 999px;
        font-size: 13px; font-weight: 700; font-family: Arial, sans-serif;}
.tablewrap {overflow-x: auto;}
table.subjects {width: 100%; border-collapse: collapse; font-size: 14px;}
table.subjects th {background: #F8F5EE !important; color: #0B2545 !important; text-align: left;
                   padding: 10px 14px; border-bottom: 2px solid #C9A227 !important;}
table.subjects td {padding: 10px 14px; border-bottom: 1px solid #eee !important;
                   color: #1b2733 !important; background: #ffffff !important;}
table.subjects tr:nth-child(even) td {background: #fbfaf6 !important;}
table.subjects td.num {color: #C9A227 !important; font-weight: 700; width: 36px;}

/* Weekly schedule */
.daygrid {display: grid; grid-template-columns: repeat(auto-fit, minmax(290px, 1fr)); gap: 12px; margin: 8px 0;}
.daycard {background: #fff; border-radius: 14px; box-shadow: 0 4px 16px rgba(0,0,0,.18); overflow: hidden;}
.daycard .dh {background: #0B2545; color: #fff !important; padding: 10px 14px; border-bottom: 3px solid #C9A227;
              font-family: Georgia, serif; font-size: 16px; display: flex; justify-content: space-between;}
.daycard.today .dh {background: #1d4e89;}
.cls {padding: 10px 14px; border-bottom: 1px solid #eee; color: #1b2733 !important; background: #fff !important;}
.cls b {color: #0B2545 !important;}
.cls .t {color: #8a6d00 !important; font-weight: 700; font-size: 13px; display: block;}
.cls small {color: #5b6b7b !important; display: block; margin-top: 2px;}
.empty {padding: 14px; color: #6b7a89 !important; background: #fff !important; font-style: italic;}
.footnote {color: #8a96a3; font-size: 12px; margin: 4px 2px;}

/* Free / busy slots */
.slotgrid {display: flex; flex-wrap: wrap; gap: 8px; padding: 14px; background: #fff;}
.slot {padding: 8px 12px; border-radius: 10px; font-size: 13px; font-weight: 600;}
.slot.free {background: #e6f6ec !important; color: #0f5132 !important; border: 1px solid #7fcf9f;}
.slot.busy {background: #fdeaea !important; color: #842029 !important; border: 1px solid #e59a9a;}

/* Notices */
.notice {padding: 12px 16px; border-radius: 12px; margin: 8px 0; font-size: 14px;}
.notice a {color: #0B2545 !important; font-weight: 700;}
.notice.ok {background: #e6f6ec !important; color: #0f5132 !important; border-left: 5px solid #2e9e5b;}
.notice.err {background: #fdeaea !important; color: #842029 !important; border-left: 5px solid #d9534f;}
.notice.info {background: #F8F5EE !important; color: #0B2545 !important; border-left: 5px solid #C9A227;}
</style>
"""

HERO = """
<div class="hero">
  <h1>🎓 Smart Campus Assistant</h1>
  <p>An AI-powered academic assistant for BS students: class schedules, lab availability,
     instructor meetings with Google Calendar, and campus complaints, in natural language.</p>
</div>
"""

display(HTML(CSS + HERO))

The HTML/CSS design
This cell defines the visual style of the whole application: a navy and gold colour theme, serif headings, a banner (HERO) with feature chips, a progress bar, large degree buttons, round semester buttons, a subjects table card, and suggested-question buttons. The last line previews the banner right here in Colab, so you can see the design before we connect it to Gradio.

In [ ]:
from IPython.display import HTML, display

CSS = """
<style>
.gradio-container {max-width: 960px !important; margin: auto !important;}
footer {display: none !important;}

.hero {background: linear-gradient(135deg, #0B2545 0%, #13315C 60%, #1d4e89 100%);
       color: #fff; padding: 28px 30px; border-radius: 18px; margin-bottom: 12px;
       border-bottom: 4px solid #C9A227; box-shadow: 0 10px 28px rgba(11,37,69,.35);}
.hero h1 {margin: 0; font-family: Georgia, 'Times New Roman', serif; font-size: 30px;
          letter-spacing: .3px; color: #fff !important;}
.hero p {margin: 8px 0 0; color: #dbe5f1 !important; line-height: 1.5;}
.chips {display: flex; flex-wrap: wrap; gap: 8px; margin-top: 16px;}
.chip {background: rgba(201,162,39,.18); border: 1px solid rgba(201,162,39,.6);
       color: #f5e7b0; padding: 6px 13px; border-radius: 999px; font-size: 13px;}

.stepper {display: flex; justify-content: space-between; gap: 6px; margin: 6px 0 12px;}
.step {flex: 1; text-align: center; font-size: 12px; opacity: .55;}
.step .dot {display: inline-flex; width: 30px; height: 30px; border-radius: 50%;
            align-items: center; justify-content: center; font-weight: 700;
            background: rgba(128,128,128,.35); margin-bottom: 4px;}
.step .lbl {display: block;}
.step.active {opacity: 1; font-weight: 700;}
.step.active .dot {background: #0B2545; color: #fff; box-shadow: 0 0 0 4px rgba(201,162,39,.55);}
.step.done {opacity: 1;}
.step.done .dot {background: #C9A227; color: #0B2545;}

.section-title {background: #F8F5EE; color: #0B2545; padding: 11px 16px; margin: 6px 0 10px;
                border-left: 5px solid #C9A227; border-radius: 8px;
                font-family: Georgia, 'Times New Roman', serif; font-size: 18px;}

.deg-btn {background: linear-gradient(135deg, #0B2545, #1d4e89) !important; color: #fff !important;
          border: none !important; border-left: 5px solid #C9A227 !important; border-radius: 12px !important;
          padding: 18px 14px !important; font-weight: 600 !important; text-align: left !important;
          transition: all .15s ease;}
.deg-btn:hover {transform: translateY(-2px); box-shadow: 0 8px 18px rgba(11,37,69,.4);}

.sem-btn {background: #F8F5EE !important; color: #0B2545 !important; border: 2px solid #C9A227 !important;
          border-radius: 999px !important; font-weight: 700 !important; transition: all .15s ease;}
.sem-btn:hover {background: #C9A227 !important; transform: translateY(-2px);}

.q-btn {background: #fff !important; color: #0B2545 !important; border: 1px solid #d9cfae !important;
        border-radius: 12px !important; text-align: left !important; font-weight: 500 !important;
        transition: all .15s ease;}
.q-btn:hover {border-color: #C9A227 !important; background: #fffaf0 !important; transform: translateX(3px);}

.link-btn {background: transparent !important; color: #C9A227 !important; border: none !important;
           text-decoration: underline; box-shadow: none !important;}

.card {background: #fff; color: #1b2733; border-radius: 14px; margin: 6px 0 14px;
       box-shadow: 0 4px 16px rgba(0,0,0,.18); overflow: hidden;}
.card-head {background: #0B2545; color: #fff; padding: 14px 18px; border-bottom: 3px solid #C9A227;
            font-family: Georgia, 'Times New Roman', serif; font-size: 18px;
            display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 8px;}
.badge {background: #C9A227; color: #0B2545; padding: 3px 12px; border-radius: 999px;
        font-size: 13px; font-weight: 700; font-family: Arial, sans-serif;}
.tablewrap {overflow-x: auto;}
table.subjects {width: 100%; border-collapse: collapse; font-size: 14px;}
table.subjects th {background: #F8F5EE !important; color: #0B2545 !important; text-align: left;
                   padding: 10px 14px; border-bottom: 2px solid #C9A227 !important;}
table.subjects td {padding: 10px 14px; border-bottom: 1px solid #eee !important;
                   color: #1b2733 !important; background: #ffffff !important;}
table.subjects tr:nth-child(even) td {background: #fbfaf6 !important;}
table.subjects td.num {color: #C9A227 !important; font-weight: 700; width: 36px;}
.card-head, .card-head span {color: #ffffff !important;}
.badge {color: #0B2545 !important;}
</style>
"""

HERO = """
<div class="hero">
  <h1>🎓 Smart Campus Assistant</h1>
  <p>An AI-powered academic assistant for BS students: class schedules, lab availability,
     instructor meetings and campus complaints, in natural language.</p>
  <div class="chips">
    <span class="chip">📅 Class schedule</span>
    <span class="chip">🧪 Lab availability</span>
    <span class="chip">👩‍🏫 Instructor meetings</span>
    <span class="chip">📝 Complaints</span>
  </div>
</div>
"""

display(HTML(CSS + HERO))

HTML helper functions
These functions create the dynamic parts of the page as HTML. stepper_html() draws the progress bar for the current stage. subjects_card() builds the table of subjects and instructors for the chosen degree and semester (using the helper functions from Step 2). greeting() creates the assistant's first chat message after a semester is chosen.

In [ ]:
import html as htmlmod
import gradio as gr

PROGRAM_CODES = list(programs.keys())
MAX_SEM_BTNS = 8
MAX_Q_BTNS = 6
STEP_NAMES = ["Degree", "Semester", "Subjects", "Ask"]
STAGE_STEP = {"degree": 0, "semester": 1, "chat": 3}

def stepper_html(stage):
    cur = STAGE_STEP[stage]
    items = []
    for i, name in enumerate(STEP_NAMES):
        cls = "done" if i < cur else ("active" if i == cur else "")
        mark = "✓" if i < cur else str(i + 1)
        items.append(f'<div class="step {cls}"><span class="dot">{mark}</span><span class="lbl">{name}</span></div>')
    return '<div class="stepper">' + "".join(items) + "</div>"

def subjects_card(program, semester):
    subjects = get_subjects(program, semester)
    rows = ""
    for i, s in enumerate(subjects, 1):
        inst = ", ".join(s["instructors"])
        rows += (f'<tr><td class="num">{i}</td><td>{htmlmod.escape(s["subject"])}</td>'
                 f'<td>{htmlmod.escape(inst)}</td></tr>')
    name = htmlmod.escape(programs[program]["program_name"])
    return f"""
    <div class="card">
      <div class="card-head"><span>📘 {name} · Semester {semester}</span>
        <span class="badge">{len(subjects)} subjects</span></div>
      <div class="tablewrap"><table class="subjects">
        <thead><tr><th>#</th><th>Subject</th><th>Instructor</th></tr></thead>
        <tbody>{rows}</tbody></table></div>
    </div>"""

def greeting(program, semester):
    return [{"role": "assistant", "content":
        f"Hello! 👋 I am your Smart Campus Assistant for **{programs[program]['program_name']}**, "
        f"semester **{semester}**.\n\nYour subjects and instructors are shown above. "
        "Tap a suggested question or type your own request."}]

Screen logic and chat handlers
The application has three screens: degree, semester and chat. render() takes the current state and returns what every component should show (visible or hidden, button labels, the subjects table, the suggested questions). The pick_* functions run when the student presses a button; they update the state and the STUDENT profile used by the tools. on_message() sends the student's message to ask_assistant() from Step 5 and adds the answer to the chat window. The conversation history is kept inside the state.

In [ ]:
def render(state, chat_value=None):
    stage, prog, sem = state["stage"], state["program"], state["semester"]

    sems = list_semesters(prog)[:MAX_SEM_BTNS] if prog else []
    sem_updates = ([gr.update(value=f"Semester {s}", visible=True) for s in sems]
                   + [gr.update(visible=False)] * (MAX_SEM_BTNS - len(sems)))

    qs = recommended_questions(prog, sem, MAX_Q_BTNS) if stage == "chat" else []
    q_updates = ([gr.update(value=q, visible=True) for q in qs]
                 + [gr.update(visible=False)] * (MAX_Q_BTNS - len(qs)))

    title = section_title(f'Step 2 · Select your semester in <b>{htmlmod.escape(programs[prog]["program_name"])}</b>') if prog else ""
    subjects = subjects_card(prog, sem) if stage == "chat" else ""
    chat_update = gr.update(value=chat_value) if chat_value is not None else gr.update()

    return ([state, stepper_html(stage),
             gr.update(visible=(stage == "degree")),
             gr.update(visible=(stage == "semester")),
             gr.update(visible=(stage == "chat")),
             title] + sem_updates + [subjects] + q_updates + [chat_update])

def pick_degree(code, state):
    state.update(stage="semester", program=code, semester=None, history=[])
    return render(state)

def pick_semester(label, state):
    sem = int(label.split()[-1])
    state.update(stage="chat", semester=sem, history=[])
    STUDENT["program"], STUDENT["semester"] = state["program"], str(sem)
    return render(state, greeting(state["program"], sem))

def back_to_degree(state):
    state.update(stage="degree", program=None, semester=None, history=[])
    return render(state, [])

def back_to_semester(state):
    state.update(stage="semester", semester=None, history=[])
    return render(state, [])

def make_degree_handler(code):
    def handler(state):
        return pick_degree(code, state)
    return handler

def on_message(text, chat, state):
    if not text or not text.strip():
        return chat, "", state
    STUDENT["program"], STUDENT["semester"] = state["program"], str(state["semester"])
    try:
        answer = ask_assistant(text.strip(), state["history"], show_calls=False)
    except Exception as e:
        answer = f"Sorry, something went wrong ({type(e).__name__}). Please try again."
    chat = chat + [{"role": "user", "content": text.strip()},
                   {"role": "assistant", "content": answer}]
    return chat, "", state

def clear_chat(state):
    state["history"] = []
    return greeting(state["program"], state["semester"]), state

# Panels for the Tabs
These cells create the content of the four feature tabs: the weekly class schedule, the lab availability view, the instructor meeting booking panel (with free slots and Google Calendar) and the complaints box.

##Date choices and weekly schedule table
date_choices() creates the date dropdown (next 14 days). weekly_schedule_html() now builds ONE proper table for the whole week: Day, Time, Subject, Instructor and Room. Each day's name appears once and spans its classes; days without classes show "No classes". Today's row is highlighted.

In [ ]:
def date_choices(days=14, skip_sunday=False):
    out = []
    for i in range(days):
        d = now() + timedelta(days=i)
        if skip_sunday and d.weekday() == 6:
            continue
        tag = " (today)" if i == 0 else (" (tomorrow)" if i == 1 else "")
        out.append((d.strftime("%a %d %b %Y") + tag, d.strftime("%Y-%m-%d")))
    return out

def weekly_schedule_html(program, semester):
    rows = [s for s in SESSIONS if s["program"] == program and s["semester"] == str(semester)]
    today_name = now().strftime("%A")
    body = ""
    for d in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]:
        items = sorted([s for s in rows if s["day"].title() == d], key=lambda s: s["start"])
        bg = "#fff3cf" if d == today_name else "#F8F5EE"
        day_style = f'style="background:{bg} !important;color:#0B2545 !important;font-weight:700;vertical-align:top;"'
        if not items:
            body += (f'<tr><td {day_style}>{d}</td>'
                     '<td colspan="4" style="color:#8a96a3 !important;font-style:italic;">No classes</td></tr>')
            continue
        for i, s in enumerate(items):
            day_td = f'<td {day_style} rowspan="{len(items)}">{d}</td>' if i == 0 else ""
            body += (f'<tr>{day_td}'
                     f'<td style="white-space:nowrap;font-weight:600;">{fmt_minutes(s["start"])} - {fmt_minutes(s["end"])}</td>'
                     f'<td>{htmlmod.escape(s["course"])}</td>'
                     f'<td>{htmlmod.escape(s["teacher"] or "TBA")}</td>'
                     f'<td>{htmlmod.escape(s["room"] or "TBA")}</td></tr>')
    name = htmlmod.escape(programs[program]["program_name"])
    return (f'<div class="card"><div class="card-head"><span>📅 Weekly schedule · {name} · Semester {semester}</span>'
            f'<span class="badge">{len(rows)} classes / week</span></div>'
            '<div class="tablewrap"><table class="subjects"><thead><tr><th>Day</th><th>Time</th><th>Subject</th>'
            f'<th>Instructor</th><th>Room</th></tr></thead><tbody>{body}</tbody></table></div></div>'
            f'<div class="footnote">End times are estimated (lecture = {THEORY_MINUTES} min, lab = {LAB_MINUTES} min).</div>')

##Banner, extra styles and colour-safe helpers
This cell replaces the banner with a version that has no feature chips (the real features are now tabs). It adds styles for the tabs, buttons and free/busy time slots. It also defines section_title() and notice(), which write their colours directly inside the HTML so Gradio's dark theme cannot hide the text.

In [ ]:
HERO = """
<div class="hero">
  <h1>🎓 Smart Campus Assistant</h1>
  <p>An AI-powered academic assistant for BS students: class schedules, lab availability,
     instructor meetings with Google Calendar, and campus complaints, in natural language.</p>
</div>
"""

try:
    CSS_BASE
except NameError:
    CSS_BASE = CSS            # keep the original CSS so this cell can be re-run safely

CSS_EXTRA = """
<style>
.go-btn {background: #0B2545 !important; color: #fff !important; border: none !important;
         border-bottom: 3px solid #C9A227 !important; border-radius: 10px !important; font-weight: 700 !important;}
.go-btn:hover {background: #1d4e89 !important;}
.warn-btn {background: #fff !important; color: #842029 !important; border: 1px solid #e59a9a !important;
           border-radius: 10px !important; font-weight: 600 !important;}
button[role="tab"] {font-weight: 600 !important; font-size: 15px !important;}
button[role="tab"][aria-selected="true"], button[role="tab"].selected {
    color: #C9A227 !important; border-bottom: 3px solid #C9A227 !important;}
.slotgrid {display: flex; flex-wrap: wrap; gap: 8px; padding: 14px; background: #fff;}
.slot {padding: 8px 12px; border-radius: 10px; font-size: 13px; font-weight: 600;}
.slot.free {background: #e6f6ec !important; color: #0f5132 !important; border: 1px solid #7fcf9f;}
.slot.busy {background: #fdeaea !important; color: #842029 !important; border: 1px solid #e59a9a;}
.footnote {color: #8a96a3; font-size: 12px; margin: 4px 2px;}
.card-head, .card-head span {color: #ffffff !important;}
.badge {background: #C9A227 !important; color: #0B2545 !important;}
</style>
"""
CSS = CSS_BASE + CSS_EXTRA

def section_title(text):
    text = text.replace("<b>", '<b style="color:#0B2545 !important">')
    return ('<div style="background:#F8F5EE !important;color:#0B2545 !important;padding:11px 16px;'
            'margin:6px 0 10px;border-left:5px solid #C9A227;border-radius:8px;'
            f'font-family:Georgia,serif;font-size:18px;">{text}</div>')

_NOTICE = {"ok": ("#e6f6ec", "#0f5132", "#2e9e5b"),
           "err": ("#fdeaea", "#842029", "#d9534f"),
           "info": ("#F8F5EE", "#0B2545", "#C9A227")}

def notice(kind, text):
    bg, fg, bar = _NOTICE[kind]
    text = text.replace("<b>", f'<b style="color:{fg} !important">')
    text = text.replace("<a ", f'<a style="color:{fg} !important;font-weight:700" ')
    return (f'<div style="background:{bg} !important;color:{fg} !important;border-left:5px solid {bar};'
            f'padding:12px 16px;border-radius:12px;margin:8px 0;font-size:14px;">{text}</div>')

display(HTML(CSS + HERO))

# Lab availability view
lab_html() shows one lab on one date as a row of hourly slots from 8:00 AM to 6:00 PM. Green means free, red means a class is using the lab (with the course name). Below the slots it lists the classes that use the lab that day.

In [ ]:
def lab_html(lab, date):
    if not lab or not date:
        return notice("info", "Choose a lab and a date to see when it is free.")
    day = datetime.strptime(date, "%Y-%m-%d").strftime("%A")
    busy = sorted([s for s in SESSIONS if s["room"] == lab and s["day"].lower() == day.lower()],
                  key=lambda s: s["start"])
    chips, free_hours = "", 0
    for h in range(8, 18):
        a, b = h * 60, (h + 1) * 60
        hit = next((s for s in busy if s["start"] < b and a < s["end"]), None)
        if hit:
            chips += f'<span class="slot busy">{fmt_minutes(a)} · busy ({htmlmod.escape(hit["course"])})</span>'
        else:
            chips += f'<span class="slot free">{fmt_minutes(a)} · free</span>'
            free_hours += 1
    classes = "".join(
        f'<tr><td>{fmt_minutes(s["start"])} - {fmt_minutes(s["end"])}</td><td>{htmlmod.escape(s["course"])}</td>'
        f'<td>{s["program"]} sem {s["semester"]}</td></tr>' for s in busy)
    table = (f'<div class="tablewrap"><table class="subjects"><thead><tr><th>Time</th><th>Class</th><th>Program</th></tr></thead>'
             f'<tbody>{classes}</tbody></table></div>') if busy else ""
    return (f'<div class="card"><div class="card-head"><span>🧪 {htmlmod.escape(lab)} · {day} {date}</span>'
            f'<span class="badge">{free_hours} free hour(s)</span></div>'
            f'<div class="slotgrid">{chips}</div>{table}</div>')

def lab_check(lab, date):
    return lab_html(lab, date)

#Meeting and complaint handlers
These functions connect the buttons to the tools from Step 3. meetings_html() and complaints_html() draw the lists, and they return an empty string when there is nothing to show (no placeholders). meet_show_slots() shows an instructor's free slots; meet_book() books the chosen slot and adds it to Google Calendar; meet_cancel() cancels a meeting, which then disappears from the list. complaint_submit() saves a complaint to complaints.json. refresh_panels() resets all tabs after the student picks a semester.

In [ ]:
UI_COMPLAINT_START = len(COMPLAINTS)     # only complaints entered in the app are shown

def meetings_html():
    active = [m for m in MEETINGS if m["status"] == "Scheduled"]
    if not active:
        return ""                          # nothing booked, so nothing is shown
    rows = "".join(
        f'<tr><td>{m["meeting_id"]}</td><td>{htmlmod.escape(m["instructor"])}</td><td>{m["date"]}</td>'
        f'<td>{m["time"]}</td><td>{htmlmod.escape(m["purpose"])}</td></tr>' for m in active)
    return (section_title("📅 My meetings")
            + '<div class="card"><div class="tablewrap"><table class="subjects"><thead><tr><th>ID</th><th>Instructor</th>'
              f'<th>Date</th><th>Time</th><th>Purpose</th></tr></thead><tbody>{rows}</tbody></table></div></div>')

def refresh_meetings():
    scheduled = [(f'{m["meeting_id"]} · {m["instructor"]} · {m["date"]} {m["time"]}', m["meeting_id"])
                 for m in MEETINGS if m["status"] == "Scheduled"]
    return meetings_html(), gr.update(choices=scheduled, value=None)

def meet_show_slots(instructor, date):
    if not instructor or not date:
        return notice("err", "Please choose an instructor and a date."), gr.update(choices=[], value=None)
    res = get_instructor_free_slots(instructor, date)
    if "error" in res:
        return notice("err", htmlmod.escape(res["error"])), gr.update(choices=[], value=None)
    if "needs_clarification" in res:
        return notice("err", "Please choose one instructor."), gr.update(choices=[], value=None)
    slots = res["free_slots"]
    if not slots:
        return notice("info", f'No free slots for {htmlmod.escape(res["instructor"])} on {res["day"]}. Try another date.'), \
               gr.update(choices=[], value=None)
    chips = "".join(f'<span class="slot free">{s["label"]}</span>' for s in slots)
    teach = (f'<div class="footnote" style="padding:0 14px 12px">Teaching that day: {htmlmod.escape("; ".join(res["teaching_that_day"]))}</div>'
             if res["teaching_that_day"] else "")
    html = (f'<div class="card"><div class="card-head"><span>🗓 {htmlmod.escape(res["instructor"])} · {res["day"]} {date}</span>'
            f'<span class="badge">{len(slots)} free slots</span></div><div class="slotgrid">{chips}</div>{teach}</div>')
    return html, gr.update(choices=[(s["label"], s["start"]) for s in slots], value=None)

def meet_book(instructor, date, slot, purpose):
    if not (instructor and date and slot):
        m_html, c_upd = refresh_meetings()
        return notice("err", "Please choose an instructor, a date and a free slot first."), gr.update(), gr.update(), m_html, c_upd
    res = book_meeting(instructor, date, slot, (purpose or "").strip() or "General discussion")
    if res.get("booked"):
        m, cal = res["meeting"], res["google_calendar"]
        if cal.get("added"):
            link = f' · <a href="{cal["link"]}" target="_blank">open event</a>' if cal.get("link") else ""
            cal_txt = f"📅 Added to your Google Calendar{link}"
        else:
            cal_txt = f'⚠️ Not added to Google Calendar ({htmlmod.escape(str(cal.get("reason", "")))})'
        msg = notice("ok", f'<b>Meeting booked ✅</b> {m["meeting_id"]} with {htmlmod.escape(m["instructor"])} '
                           f'on {m["date"]}, {m["time"]}.<br>{cal_txt}')
    elif "needs_clarification" in res:
        msg = notice("err", "Please choose one instructor.")
    else:
        msg = notice("err", htmlmod.escape(res.get("error", "Could not book the meeting.")))
    s_html, s_upd = meet_show_slots(instructor, date)
    m_html, c_upd = refresh_meetings()
    return msg, s_html, s_upd, m_html, c_upd

def meet_cancel(meeting_id):
    if not meeting_id:
        m_html, c_upd = refresh_meetings()
        return notice("err", "Select a meeting to cancel."), m_html, c_upd
    res = cancel_meeting(meeting_id)
    if res.get("cancelled"):
        extra = " The Google Calendar event was removed." if res.get("calendar_event_removed") else ""
        msg = notice("ok", f'Meeting {res["meeting"]["meeting_id"]} cancelled.{extra}')
    else:
        msg = notice("err", htmlmod.escape(res.get("error", "Could not cancel.")))
    m_html, c_upd = refresh_meetings()
    return msg, m_html, c_upd

def complaints_html():
    mine = COMPLAINTS[UI_COMPLAINT_START:]
    if not mine:
        return ""                          # nothing entered, so nothing is shown
    rows = "".join(
        f'<tr><td>{c["complaint_id"]}</td><td>{htmlmod.escape(c["category"])}</td>'
        f'<td>{htmlmod.escape(c["description"])}</td><td>{c["submitted_at"]}</td><td>{c["status"]}</td></tr>'
        for c in mine[::-1])
    return (section_title("📝 Your complaints")
            + '<div class="card"><div class="tablewrap"><table class="subjects"><thead><tr><th>ID</th><th>Category</th>'
              f'<th>Description</th><th>Submitted</th><th>Status</th></tr></thead><tbody>{rows}</tbody></table></div></div>')

def complaint_submit(category, text):
    res = submit_complaint(category or "Other", text or "")
    if res.get("submitted"):
        msg = notice("ok", f'<b>Complaint {res["complaint_id"]} submitted ✅</b> Category: {htmlmod.escape(res["category"])}. '
                           'Status: Submitted. It has been saved to complaints.json.')
        return msg, complaints_html(), gr.update(value="")
    return notice("err", htmlmod.escape(res.get("error", "Could not submit."))), complaints_html(), gr.update()

def refresh_panels(state):
    prog, sem = state["program"], state["semester"]
    if not prog or not sem:
        return [gr.update()] * 8
    m_html, c_upd = refresh_meetings()
    return [weekly_schedule_html(prog, sem),
            gr.update(choices=unique_instructors(prog, sem), value=None),
            "",                                   # slots_html: empty until slots are requested
            gr.update(choices=[], value=None),
            "",                                   # meet_msg
            "",                                   # lab_out
            m_html, c_upd]

#Build and launch the Gradio application
Tabs now sit at the top of the chat screen. The subjects table moved inside the Assistant tab, followed by the suggested questions and the chat. At launch the list of meetings is cleared and only complaints entered in this app are shown, so no test data appears. Headings of the meetings and complaints lists are part of the lists themselves, so nothing shows until the student adds something.

In [ ]:
MEETINGS.clear()                        # remove test meetings left by earlier cells
UI_COMPLAINT_START = len(COMPLAINTS)    # hide earlier (test) complaints

DATES = date_choices()
DATES_NO_SUN = date_choices(skip_sunday=True)

with gr.Blocks(title="Smart Campus Assistant") as demo:
    gr.HTML(CSS + HERO)
    stepper = gr.HTML(stepper_html("degree"))
    state = gr.State({"stage": "degree", "program": None, "semester": None, "history": []})

    # Screen 1: degree
    with gr.Column(visible=True) as degree_col:
        gr.HTML(section_title("Step 1 · Select your degree program"))
        with gr.Row():
            degree_btns = [gr.Button(f"{c} · {programs[c]['program_name']}",
                                     elem_classes=["deg-btn"], min_width=200)
                           for c in PROGRAM_CODES]

    # Screen 2: semester
    with gr.Column(visible=False) as sem_col:
        sem_title = gr.HTML()
        with gr.Row():
            sem_btns = [gr.Button("", visible=False, elem_classes=["sem-btn"], min_width=110)
                        for _ in range(MAX_SEM_BTNS)]
        back_deg = gr.Button("← Change degree", elem_classes=["link-btn"])

    # Screen 3: tabs (on top)
    with gr.Column(visible=False) as chat_col:
        with gr.Tabs():
            # Tab 1: subjects + suggested questions + chat
            with gr.Tab("💬 Assistant"):
                subjects_out = gr.HTML()
                gr.HTML(section_title("💡 Suggested questions"))
                q_btns = [gr.Button("", visible=False, elem_classes=["q-btn"]) for _ in range(MAX_Q_BTNS)]
                try:
                    chatbot = gr.Chatbot(type="messages", height=380, show_label=False)
                except TypeError:
                    chatbot = gr.Chatbot(height=380, show_label=False)
                box = gr.Textbox(placeholder="Ask about your schedule, labs, meetings or complaints, then press Enter...",
                                 show_label=False)
                with gr.Row():
                    back_sem = gr.Button("← Change semester", elem_classes=["link-btn"])
                    clear_btn = gr.Button("🧹 Clear chat", elem_classes=["link-btn"])

            # Tab 2: weekly schedule
            with gr.Tab("📅 Class Schedule"):
                schedule_out = gr.HTML()

            # Tab 3: lab availability
            with gr.Tab("🧪 Lab Availability"):
                with gr.Row():
                    lab_dd = gr.Dropdown(LABS, label="Lab", value=LABS[0] if LABS else None)
                    lab_date = gr.Dropdown(DATES, label="Date", value=DATES[1][1])
                lab_btn = gr.Button("🔎 Check availability", elem_classes=["go-btn"])
                lab_out = gr.HTML()

            # Tab 4: instructor meeting
            with gr.Tab("👩‍🏫 Instructor Meeting"):
                gr.HTML(section_title("Book a 30-minute meeting"))
                with gr.Row():
                    meet_inst = gr.Dropdown([], label="Instructor")
                    meet_date = gr.Dropdown(DATES_NO_SUN, label="Date", value=DATES_NO_SUN[1][1])
                show_btn = gr.Button("🔎 Show free slots", elem_classes=["go-btn"])
                slots_html = gr.HTML()
                slot_radio = gr.Radio([], label="Select a free slot")
                purpose = gr.Textbox(label="Purpose (optional)", placeholder="e.g. FYP discussion")
                book_btn = gr.Button("📅 Book meeting and add to Google Calendar", elem_classes=["go-btn"])
                meet_msg = gr.HTML()
                meetings_out = gr.HTML()          # heading and table appear only after a booking
                with gr.Row():
                    cancel_dd = gr.Dropdown([], label="Select a meeting to cancel")
                    cancel_btn = gr.Button("❌ Cancel meeting", elem_classes=["warn-btn"])
                    refresh_btn = gr.Button("🔄 Refresh list", elem_classes=["link-btn"])

            # Tab 5: complaints
            with gr.Tab("📝 Complaints"):
                gr.HTML(section_title("Submit a complaint"))
                c_cat = gr.Dropdown(COMPLAINT_CATEGORIES, label="Category", value=COMPLAINT_CATEGORIES[0])
                c_text = gr.Textbox(label="Describe the problem", lines=4,
                                    placeholder="Write what happened, where and when...")
                c_btn = gr.Button("📨 Submit complaint", elem_classes=["go-btn"])
                c_msg = gr.HTML()
                c_table = gr.HTML(complaints_html())   # empty until a complaint is submitted

    OUTS = ([state, stepper, degree_col, sem_col, chat_col, sem_title]
            + sem_btns + [subjects_out] + q_btns + [chatbot])
    PANEL_OUTS = [schedule_out, meet_inst, slots_html, slot_radio, meet_msg, lab_out, meetings_out, cancel_dd]

    # navigation
    for code, b in zip(PROGRAM_CODES, degree_btns):
        b.click(make_degree_handler(code), [state], OUTS)
    for b in sem_btns:
        b.click(pick_semester, [b, state], OUTS).then(refresh_panels, [state], PANEL_OUTS)
    back_deg.click(back_to_degree, [state], OUTS)
    back_sem.click(back_to_semester, [state], OUTS)

    # chat
    for b in q_btns:
        b.click(on_message, [b, chatbot, state], [chatbot, box, state])
    box.submit(on_message, [box, chatbot, state], [chatbot, box, state])
    clear_btn.click(clear_chat, [state], [chatbot, state])

    # tabs
    lab_btn.click(lab_check, [lab_dd, lab_date], [lab_out])
    show_btn.click(meet_show_slots, [meet_inst, meet_date], [slots_html, slot_radio])
    book_btn.click(meet_book, [meet_inst, meet_date, slot_radio, purpose],
                   [meet_msg, slots_html, slot_radio, meetings_out, cancel_dd])
    cancel_btn.click(meet_cancel, [cancel_dd], [meet_msg, meetings_out, cancel_dd])
    refresh_btn.click(refresh_meetings, [], [meetings_out, cancel_dd])
    c_btn.click(complaint_submit, [c_cat, c_text], [c_msg, c_table, c_text])

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ed1c2654e2b4403287.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
